# Asymmetric Recommendation Matrix

This version preserves the original notebook and writes only `*_asymmetric` artifacts.

The directional score is a Bayesian-smoothed estimate of `P(target liked | source liked, both games reviewed)`. Therefore, `A → B` and `B → A` can receive different scores and ranks.

Run the notebook from top to bottom. The final cell creates `game_recommendation_lists_asymmetric.parquet`, matching the API serving schema without replacing the current production artifact.


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet("processed_reviews.parquet")

In [ ]:
df.columns

In [ ]:
appids = df.appid.unique()
users= df.author_steamid.unique()

In [ ]:
from pathlib import Path
import threading
import time

import duckdb
import pandas as pd


# ============================================================
# MAJOR PARAMETERS
# ============================================================

EDGE_OUTPUT_FILE = "game_edges_asymmetric.parquet"
GAME_LOOKUP_FILE = "game_lookup_asymmetric.parquet"
DATABASE_FILE = "recommendation_build_asymmetric.duckdb"
TEMP_DIRECTORY = "duckdb_temp_asymmetric"

USER_COLUMN = "author_steamid"
GAME_COLUMN = "appid"
GAME_NAME_COLUMN = "game"
VOTE_COLUMN = "voted_up"
TIMESTAMP_COLUMN = "timestamp_updated"

# Adjust these for your computer
MEMORY_LIMIT = "80GB"
THREADS = 16

# Keep all observed edges for now.
# Weak edges can be removed later without rebuilding.
MIN_SHARED_REVIEWS = 1

# Progress update frequency
PROGRESS_POLL_SECONDS = 2.0


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIRECTORY = Path.cwd()

edge_output_path = (
    PROJECT_DIRECTORY / EDGE_OUTPUT_FILE
).as_posix()

game_lookup_path = (
    PROJECT_DIRECTORY / GAME_LOOKUP_FILE
).as_posix()

database_path = (
    PROJECT_DIRECTORY / DATABASE_FILE
).as_posix()

temp_path = (
    PROJECT_DIRECTORY / TEMP_DIRECTORY
).as_posix()

Path(temp_path).mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def directory_size_gb(directory: str) -> float:
    """
    Return the total size of a directory in GiB.
    """

    directory_path = Path(directory)

    if not directory_path.exists():
        return 0.0

    total_bytes = sum(
        file_path.stat().st_size
        for file_path in directory_path.rglob("*")
        if file_path.is_file()
    )

    return total_bytes / 1024**3


def run_query_with_progress(
    connection: duckdb.DuckDBPyConnection,
    sql: str,
    description: str,
    temp_directory: str,
    poll_seconds: float = 2.0,
) -> None:
    """
    Run a DuckDB query in a background thread while printing
    progress information in the main thread.

    This is useful in VS Code, where DuckDB's native terminal
    progress bar may not display.
    """

    result = {
        "error": None,
    }

    def execute_query() -> None:
        try:
            connection.execute(sql)

        except Exception as error:
            result["error"] = error

    worker = threading.Thread(
        target=execute_query,
        daemon=True,
    )

    start_time = time.perf_counter()
    worker.start()

    last_print_time = 0.0

    while worker.is_alive():
        current_time = time.perf_counter()

        if current_time - last_print_time >= poll_seconds:
            elapsed_seconds = current_time - start_time
            elapsed_minutes = elapsed_seconds / 60

            try:
                progress = connection.query_progress()
            except Exception:
                progress = None

            temp_size = directory_size_gb(temp_directory)

            if progress is not None and progress >= 0:
                # DuckDB versions may expose either 0–1 or 0–100.
                if progress <= 1:
                    progress_percent = progress * 100
                else:
                    progress_percent = progress

                progress_text = f"{progress_percent:5.1f}%"

            else:
                progress_text = "working"

            print(
                f"{description}: "
                f"{progress_text} | "
                f"{elapsed_minutes:,.1f} min elapsed | "
                f"temp disk: {temp_size:,.2f} GiB",
                flush=True,
            )

            last_print_time = current_time

        time.sleep(0.25)

    worker.join()

    elapsed_minutes = (
        time.perf_counter() - start_time
    ) / 60

    if result["error"] is not None:
        print(
            f"{description}: FAILED after "
            f"{elapsed_minutes:,.1f} minutes."
        )

        raise result["error"]

    print(
        f"{description}: complete in "
        f"{elapsed_minutes:,.1f} minutes."
    )


# ============================================================
# VALIDATE THE EXISTING DATAFRAME
# ============================================================

required_columns = {
    USER_COLUMN,
    GAME_COLUMN,
    GAME_NAME_COLUMN,
    VOTE_COLUMN,
    TIMESTAMP_COLUMN,
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

print("Input DataFrame validated.")
print(f"Input rows: {len(df):,}")


# ============================================================
# CONNECT TO DUCKDB
# ============================================================

con = duckdb.connect(database_path)

try:
    con.execute(
        f"SET memory_limit = '{MEMORY_LIMIT}'"
    )

    con.execute(
        f"SET threads = {THREADS}"
    )

    con.execute(
        f"SET temp_directory = '{temp_path}'"
    )

    # The custom progress function is used instead.
    con.execute(
        "SET enable_progress_bar = false"
    )

    # Make the existing pandas DataFrame available in DuckDB.
    con.register(
        "source_df",
        df,
    )

    print()
    print("DuckDB connected.")
    print(f"Memory limit: {MEMORY_LIMIT}")
    print(f"Threads: {THREADS}")
    print(f"Temporary directory: {temp_path}")


    # ========================================================
    # STAGE 1: CREATE LEAN REVIEW TABLE
    # ========================================================
    #
    # This removes the huge review text and all unrelated
    # columns before generating game pairs.
    #
    # If the same user reviewed the same app more than once,
    # retain the newest review.
    # ========================================================

    print()
    print("Stage 1/3: Creating lean review table...")

    stage_start = time.perf_counter()

    con.execute("DROP TABLE IF EXISTS reviews")

    con.execute(f"""
        CREATE TABLE reviews AS

        SELECT
            CAST("{USER_COLUMN}" AS UBIGINT)
                AS author_steamid,

            CAST("{GAME_COLUMN}" AS UINTEGER)
                AS appid,

            CAST("{VOTE_COLUMN}" AS BOOLEAN)
                AS voted_up

        FROM source_df

        WHERE "{USER_COLUMN}" IS NOT NULL
          AND "{GAME_COLUMN}" IS NOT NULL
          AND "{VOTE_COLUMN}" IS NOT NULL

        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY
                "{USER_COLUMN}",
                "{GAME_COLUMN}"

            ORDER BY
                "{TIMESTAMP_COLUMN}" DESC
        ) = 1
    """)

    stage_elapsed = (
        time.perf_counter() - stage_start
    ) / 60

    lean_review_count = con.execute("""
        SELECT COUNT(*)
        FROM reviews
    """).fetchone()[0]

    print(
        f"Stage 1 complete in "
        f"{stage_elapsed:,.1f} minutes."
    )

    print(
        f"Lean review rows: "
        f"{lean_review_count:,}"
    )


    # ========================================================
    # STAGE 2: BUILD APPID → GAME LOOKUP
    # ========================================================
    #
    # If one appid has multiple observed names, choose the most
    # frequently occurring name.
    # ========================================================

    print()
    print("Stage 2/3: Building game-name lookup...")

    stage_start = time.perf_counter()

    con.execute(f"""
        COPY (
            WITH name_counts AS (
                SELECT
                    CAST("{GAME_COLUMN}" AS UINTEGER)
                        AS appid,

                    CAST("{GAME_NAME_COLUMN}" AS VARCHAR)
                        AS game,

                    COUNT(*) AS name_frequency

                FROM source_df

                WHERE "{GAME_COLUMN}" IS NOT NULL
                  AND "{GAME_NAME_COLUMN}" IS NOT NULL

                GROUP BY
                    "{GAME_COLUMN}",
                    "{GAME_NAME_COLUMN}"
            ),

            ranked_names AS (
                SELECT
                    appid,
                    game,
                    name_frequency,

                    ROW_NUMBER() OVER (
                        PARTITION BY appid

                        ORDER BY
                            name_frequency DESC,
                            game
                    ) AS name_rank

                FROM name_counts
            )

            SELECT
                appid,
                game

            FROM ranked_names

            WHERE name_rank = 1
        )

        TO '{game_lookup_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    stage_elapsed = (
        time.perf_counter() - stage_start
    ) / 60

    print(
        f"Stage 2 complete in "
        f"{stage_elapsed:,.1f} minutes."
    )

    print(
        f"Created: {GAME_LOOKUP_FILE}"
    )


    # ========================================================
    # STAGE 3: BUILD SPARSE GAME EDGES
    # ========================================================
    #
    # a.appid < b.appid guarantees:
    #
    # 1. No game is paired with itself.
    # 2. A-B and B-A are not both created.
    # 3. appid_a is always smaller than appid_b.
    #
    # Every user who reviewed both games increments shared_review_count.
    # The directional denominators count shared reviewers who liked A or B.
    # This supports P(B liked | A liked, both reviewed) separately from
    # P(A liked | B liked, both reviewed).
    # ========================================================

    edge_sql = f"""
        COPY (
            SELECT
                a.appid AS appid_a,
                b.appid AS appid_b,

                SUM(
                    CASE
                        WHEN a.voted_up
                         AND b.voted_up
                        THEN 1
                        ELSE 0
                    END
                )::UBIGINT
                    AS both_positive_count,

                COUNT(*)::UBIGINT
                    AS shared_review_count,

                SUM(
                    CASE WHEN a.voted_up THEN 1 ELSE 0 END
                )::UBIGINT AS a_positive_count,

                SUM(
                    CASE WHEN b.voted_up THEN 1 ELSE 0 END
                )::UBIGINT AS b_positive_count,

                SUM(
                    CASE
                        WHEN a.voted_up
                         AND b.voted_up
                        THEN 1
                        ELSE 0
                    END
                )::DOUBLE
                / COUNT(*)
                    AS symmetric_raw_score

            FROM reviews AS a

            INNER JOIN reviews AS b
                ON a.author_steamid
                 = b.author_steamid

               AND a.appid
                 < b.appid

            GROUP BY
                a.appid,
                b.appid

            HAVING COUNT(*) >= {MIN_SHARED_REVIEWS}
        )

        TO '{edge_output_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """

    print()
    print("Stage 3/3: Building sparse game edges...")
    print("This is the main long-running step.")

    run_query_with_progress(
        connection=con,
        sql=edge_sql,
        description="Building game edges",
        temp_directory=temp_path,
        poll_seconds=PROGRESS_POLL_SECONDS,
    )


    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print()
    print("Calculating final summary...")

    user_count = con.execute("""
        SELECT COUNT(DISTINCT author_steamid)
        FROM reviews
    """).fetchone()[0]

    game_count = con.execute("""
        SELECT COUNT(DISTINCT appid)
        FROM reviews
    """).fetchone()[0]

    edge_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{edge_output_path}')
    """).fetchone()[0]

    edge_file_size = (
        Path(edge_output_path).stat().st_size
        / 1024**3
    )

    lookup_file_size = (
        Path(game_lookup_path).stat().st_size
        / 1024**2
    )

    print()
    print("=" * 60)
    print("BUILD COMPLETE")
    print("=" * 60)

    print(
        f"Reviews used:       "
        f"{lean_review_count:,}"
    )

    print(
        f"Users:              "
        f"{user_count:,}"
    )

    print(
        f"Games:              "
        f"{game_count:,}"
    )

    print(
        f"Unique game edges:  "
        f"{edge_count:,}"
    )

    print(
        f"Edge file size:     "
        f"{edge_file_size:,.2f} GiB"
    )

    print(
        f"Lookup file size:   "
        f"{lookup_file_size:,.2f} MiB"
    )

    print()
    print(
        f"Created: {EDGE_OUTPUT_FILE}"
    )

    print(
        f"Created: {GAME_LOOKUP_FILE}"
    )

finally:
    try:
        con.unregister("source_df")
    except Exception:
        pass

    con.close()


In [ ]:
from pathlib import Path
import duckdb


# ============================================================
# MAJOR PARAMETERS
# ============================================================

INPUT_EDGE_FILE = "game_edges_asymmetric.parquet"
OUTPUT_EDGE_FILE = "game_edges_asymmetric_scored.parquet"

# Neutral Beta-prior mean used for both directions. A later model version can
# replace this with target-specific positive-rate priors.
PRIOR_MEAN = 0.50
PRIOR_STRENGTH = 100

MEMORY_LIMIT = "80GB"
THREADS = 16


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path.cwd()
input_path = (PROJECT_DIR / INPUT_EDGE_FILE).as_posix()
output_path = (PROJECT_DIR / OUTPUT_EDGE_FILE).as_posix()


# ============================================================
# CREATE DIRECTIONALLY SCORED EDGE FILE
# ============================================================

con = duckdb.connect()

try:
    con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
    con.execute(f"SET threads = {THREADS}")

    print("Calculating asymmetric Bayesian-smoothed scores...")

    con.execute(f"""
        COPY (
            SELECT
                appid_a,
                appid_b,
                both_positive_count,
                shared_review_count,
                a_positive_count,
                b_positive_count,
                symmetric_raw_score,

                both_positive_count::DOUBLE
                    / NULLIF(a_positive_count, 0)
                    AS a_to_b_raw_score,

                both_positive_count::DOUBLE
                    / NULLIF(b_positive_count, 0)
                    AS b_to_a_raw_score,

                (
                    both_positive_count
                    + {PRIOR_MEAN} * {PRIOR_STRENGTH}
                )
                /
                (
                    a_positive_count
                    + {PRIOR_STRENGTH}
                ) AS a_to_b_smoothed_score,

                (
                    both_positive_count
                    + {PRIOR_MEAN} * {PRIOR_STRENGTH}
                )
                /
                (
                    b_positive_count
                    + {PRIOR_STRENGTH}
                ) AS b_to_a_smoothed_score

            FROM read_parquet('{input_path}')
        )
        TO '{output_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

finally:
    con.close()

print(f"Created: {OUTPUT_EDGE_FILE}")


In [ ]:
import duckdb
import pandas as pd


# ============================================================
# PARAMETERS
# ============================================================

EDGE_FILE = "game_edges_asymmetric_scored.parquet"
LOOKUP_FILE = "game_lookup_asymmetric.parquet"

# Use either TARGET_APPID or GAME_SEARCH.
# When TARGET_APPID is not None, it takes priority.
TARGET_APPID = 292030
GAME_SEARCH = ""

MIN_SHARED_REVIEWS = 5
RESULT_LIMIT = 30


# ============================================================
# RESOLVE THE SOURCE GAME
# ============================================================

con = duckdb.connect()

if TARGET_APPID is None:
    matches = con.execute(
        """
        SELECT appid, game
        FROM read_parquet(?)
        WHERE game ILIKE ?
        ORDER BY
            CASE
                WHEN lower(game) = lower(?) THEN 0
                WHEN lower(game) LIKE lower(?) THEN 1
                ELSE 2
            END,
            length(game),
            game
        LIMIT 20
        """,
        [LOOKUP_FILE, f"%{GAME_SEARCH}%", GAME_SEARCH, f"{GAME_SEARCH}%"],
    ).df()

    if matches.empty:
        con.close()
        raise ValueError(f"No game found matching: {GAME_SEARCH!r}")

    print("Matching games:")
    display(matches)
    TARGET_APPID = int(matches.iloc[0]["appid"])
    target_game = matches.iloc[0]["game"]
else:
    target_match = con.execute(
        """
        SELECT game
        FROM read_parquet(?)
        WHERE appid = ?
        LIMIT 1
        """,
        [LOOKUP_FILE, TARGET_APPID],
    ).fetchone()
    target_game = target_match[0] if target_match is not None else "Unknown game"

print()
print(f"Recommendations for: {target_game}")
print(f"App ID: {TARGET_APPID:,}")


# ============================================================
# QUERY THE CORRECT SCORE FOR EACH DIRECTION
# ============================================================

recommendations = con.execute(
    """
    WITH connections AS (
        -- Selected game is A: use A -> B.
        SELECT
            edges.appid_b AS connected_appid,
            edges.a_positive_count AS source_positive_count,
            edges.b_positive_count AS target_positive_count,
            edges.both_positive_count,
            edges.shared_review_count,
            edges.a_to_b_raw_score AS raw_score,
            edges.a_to_b_smoothed_score AS smoothed_score
        FROM read_parquet(?) AS edges
        WHERE edges.appid_a = ?

        UNION ALL

        -- Selected game is B: use B -> A.
        SELECT
            edges.appid_a AS connected_appid,
            edges.b_positive_count AS source_positive_count,
            edges.a_positive_count AS target_positive_count,
            edges.both_positive_count,
            edges.shared_review_count,
            edges.b_to_a_raw_score AS raw_score,
            edges.b_to_a_smoothed_score AS smoothed_score
        FROM read_parquet(?) AS edges
        WHERE edges.appid_b = ?
    )
    SELECT
        connections.connected_appid AS appid,
        lookup.game,
        connections.source_positive_count,
        connections.target_positive_count,
        connections.both_positive_count,
        connections.shared_review_count,
        connections.raw_score,
        connections.smoothed_score
    FROM connections
    LEFT JOIN read_parquet(?) AS lookup
        ON connections.connected_appid = lookup.appid
    WHERE connections.shared_review_count >= ?
    ORDER BY
        connections.smoothed_score DESC,
        connections.shared_review_count DESC,
        connections.connected_appid
    LIMIT ?
    """,
    [
        EDGE_FILE,
        TARGET_APPID,
        EDGE_FILE,
        TARGET_APPID,
        LOOKUP_FILE,
        MIN_SHARED_REVIEWS,
        RESULT_LIMIT,
    ],
).df()

con.close()
recommendations


In [ ]:
from pathlib import Path
import duckdb


# ============================================================
# PARAMETERS
# ============================================================

EDGE_FILE = "game_edges_asymmetric_scored.parquet"
OUTPUT_FILE = "game_recommendations_asymmetric.parquet"

TOP_N = 50
MIN_SHARED_REVIEWS = 5

MEMORY_LIMIT = "80GB"
THREADS = 16

PROJECT_DIR = Path.cwd()
edge_path = (PROJECT_DIR / EDGE_FILE).as_posix()
output_path = (PROJECT_DIR / OUTPUT_FILE).as_posix()


# ============================================================
# BUILD DIRECTED RECOMMENDATION TABLE
# ============================================================

con = duckdb.connect()

try:
    con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
    con.execute(f"SET threads = {THREADS}")

    print("Expanding edges with direction-specific scores...")

    con.execute(f"""
        COPY (
            WITH directed_edges AS (
                -- A recommends B using P(B+ | A+, both reviewed).
                SELECT
                    appid_a AS source_appid,
                    appid_b AS recommended_appid,
                    a_positive_count AS source_positive_count,
                    b_positive_count AS target_positive_count,
                    both_positive_count,
                    shared_review_count,
                    a_to_b_raw_score AS raw_score,
                    a_to_b_smoothed_score AS smoothed_score
                FROM read_parquet('{edge_path}')
                WHERE shared_review_count >= {MIN_SHARED_REVIEWS}

                UNION ALL

                -- B recommends A using P(A+ | B+, both reviewed).
                SELECT
                    appid_b AS source_appid,
                    appid_a AS recommended_appid,
                    b_positive_count AS source_positive_count,
                    a_positive_count AS target_positive_count,
                    both_positive_count,
                    shared_review_count,
                    b_to_a_raw_score AS raw_score,
                    b_to_a_smoothed_score AS smoothed_score
                FROM read_parquet('{edge_path}')
                WHERE shared_review_count >= {MIN_SHARED_REVIEWS}
            ),
            ranked AS (
                SELECT
                    *,
                    ROW_NUMBER() OVER (
                        PARTITION BY source_appid
                        ORDER BY
                            smoothed_score DESC,
                            shared_review_count DESC,
                            recommended_appid
                    ) AS recommendation_rank
                FROM directed_edges
            )
            SELECT
                source_appid,
                recommended_appid,
                recommendation_rank,
                smoothed_score AS score,
                raw_score,
                source_positive_count,
                target_positive_count,
                shared_review_count,
                both_positive_count
            FROM ranked
            WHERE recommendation_rank <= {TOP_N}
            ORDER BY source_appid, recommendation_rank
        )
        TO '{output_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    row_count = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
    ).fetchone()[0]
    game_count = con.execute(
        f"SELECT COUNT(DISTINCT source_appid) FROM read_parquet('{output_path}')"
    ).fetchone()[0]

    print("Asymmetric recommendation table complete.")
    print(f"Rows:  {row_count:,}")
    print(f"Games: {game_count:,}")
    print(f"File:  {OUTPUT_FILE}")

finally:
    con.close()


In [ ]:
import duckdb

TARGET_APPID = 433340
RESULT_LIMIT = 10

con = duckdb.connect()

recommendations = con.execute(
    """
    SELECT
        r.recommended_appid AS appid,
        l.game,
        r.recommendation_rank,
        r.score,
        r.raw_score,
        r.source_positive_count,
        r.target_positive_count,
        r.shared_review_count,
        r.both_positive_count
    FROM read_parquet('game_recommendations_asymmetric.parquet') AS r
    LEFT JOIN read_parquet('game_lookup_asymmetric.parquet') AS l
        ON r.recommended_appid = l.appid
    WHERE r.source_appid = ?
    ORDER BY r.recommendation_rank
    LIMIT ?
    """,
    [TARGET_APPID, RESULT_LIMIT],
).df()

con.close()
recommendations


In [ ]:
from pathlib import Path
import duckdb


# ============================================================
# BUILD THE COMPACT SERVING ARTIFACT
# ============================================================

INPUT_FILE = "game_recommendations_asymmetric.parquet"
OUTPUT_FILE = "game_recommendation_lists_asymmetric.parquet"
SERVING_TOP_N = 30

PROJECT_DIR = Path.cwd()
input_path = (PROJECT_DIR / INPUT_FILE).as_posix()
output_path = (PROJECT_DIR / OUTPUT_FILE).as_posix()

con = duckdb.connect()

try:
    con.execute(f"""
        COPY (
            SELECT
                CAST(source_appid AS UINTEGER) AS appid,
                CAST(
                    list(
                        recommended_appid
                        ORDER BY recommendation_rank
                    )
                    AS UINTEGER[]
                ) AS recommendations
            FROM read_parquet('{input_path}')
            WHERE recommendation_rank <= {SERVING_TOP_N}
            GROUP BY source_appid
            ORDER BY source_appid
        )
        TO '{output_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    schema = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{output_path}')"
    ).df()
    row_count = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
    ).fetchone()[0]

    print("Compact asymmetric serving artifact complete.")
    print(f"Sources: {row_count:,}")
    print(f"File:    {OUTPUT_FILE}")
    display(schema)

finally:
    con.close()
